# E01 Matrix Sensing Benchmark - PyTorch

This notebook is the PyTorch notebook-first rewrite of the Matrix Sensing benchmark. It keeps the experiment visible in one place:

- target matrix generation
- measurement tensor generation
- matrix sensing loss
- optimizer selection
- explicit `step` function binding
- process-parallel run loop
- metrics
- plots
- conclusion

Official `torch.optim.Muon` is used when available; local exact-SVD Muon and Shampoo helpers come from `optimizers/`.

Default mode is the full E01 grid: 5 optimizers, 3 dimensions, 10 seeds, and 2000 iterations per run. Quick validation lives outside the notebook in `Smoketest/`. Results stay in notebook memory as `df` and `trajectories`; this notebook does not write CSV, PNG, or report files.

In [ ]:
from pathlib import Path
import os
import sys
from functools import partial
from inspect import getsource
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import get_context

for thread_env in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ.setdefault(thread_env, "1")

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display
from tqdm.auto import tqdm

PROJECT = Path.cwd().resolve()
if not (PROJECT / "problems").exists():
    PROJECT = PROJECT.parent.resolve()
sys.path.insert(0, str(PROJECT))

from problems.MatrixSensing import run_spec as run_matrix_sensing_spec, step

torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

print(f"project = {PROJECT}")
print(f"torch   = {torch.__version__}")

## Parameters And Full Run Grid

This cell defines both the experiment constants and the exact per-run parameter dictionaries used by the process pool. `GRID_PREVIEW` is one row per run, not a grouped summary.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

ALGOS = ["Muon", "Muon-Exact", "Shampoo", "Adam", "SGD"]
DIMS = [50, 60, 70]
RANK = 5
LR = 0.01
NOISE = 0.0
DIST = "normal"
SPECTRUM = "hard-cutoff"
KAPPA = 1.0
INIT_SCALE = 0.01
ITERS = 2000
SEEDS = list(range(10))
EPSILON = 0.01
NUM_WORKERS = min(8, os.cpu_count() or 1)

RUN_SPECS = [
    {
        "algo": algo,
        "d": d,
        "rank": RANK,
        "lr": LR,
        "noise": NOISE,
        "dist": DIST,
        "spectrum": SPECTRUM,
        "kappa": KAPPA,
        "init_scale": INIT_SCALE,
        "seed": seed,
        "iters": ITERS,
        "epsilon": EPSILON,
        "device_type": DEVICE.type,
        "dtype_name": "float64",
    }
    for algo in ALGOS
    for d in DIMS
    for seed in SEEDS
]

GRID_PREVIEW = pd.DataFrame(RUN_SPECS)
GRID_PREVIEW.insert(3, "m_meas", 2 * GRID_PREVIEW["d"] * GRID_PREVIEW["rank"])
GRID_PREVIEW["run_id"] = range(len(GRID_PREVIEW))
GRID_PREVIEW = GRID_PREVIEW[
    [
        "run_id",
        "algo",
        "d",
        "rank",
        "m_meas",
        "seed",
        "lr",
        "iters",
        "epsilon",
        "noise",
        "dist",
        "spectrum",
        "kappa",
        "init_scale",
        "device_type",
        "dtype_name",
    ]
]

print(f"device={DEVICE}, dtype={DTYPE}")
print(f"workers={NUM_WORKERS}")
print(f"runs={len(RUN_SPECS)}, iters={ITERS}, total_steps={len(RUN_SPECS) * ITERS}")
display(GRID_PREVIEW)

## Matrix Sensing Problem

Target matrix:

$$
X^\star = U \operatorname{diag}(s)V^\top
$$

Measurements:

$$
y_i = \langle A_i, X^\star\rangle + \varepsilon_i
$$

Loss:

$$
f(X) = \frac{1}{2m}\sum_{i=1}^{m}(\langle A_i, X\rangle-y_i)^2
$$

The key PyTorch expression is `torch.einsum("mij,ij->m", A, X)`, which computes all matrix inner products at once.

The per-run implementation lives in `problems.MatrixSensing.run_spec` and is imported as `run_matrix_sensing_spec`. This keeps each worker function importable for `ProcessPoolExecutor` with the `spawn` start method.

## Optimizers

`Muon` uses official `torch.optim.Muon` when available and falls back to local Newton-Schulz Muon on older PyTorch; `Muon-Exact` uses the exact-SVD `optimizers/MuonExact.py`; `Shampoo` uses the matrix-only implementation in `optimizers/Shampoo.py`; `Adam` and `SGD` use PyTorch built-ins.

Optimizer construction is part of the importable worker. The compared methods are the names listed in `ALGOS` in the parameter cell.

## Single Run Worker

Each worker returns one result row and one trajectory. `K_epsilon` is recorded when the loss first crosses the threshold, but the loop does not break early, so every run has comparable full-length trajectories.

In [ ]:
# This notebook-visible function object is passed into the worker runner.
# It remains multiprocessing-safe because it is imported from `problems.MatrixSensing`.
STEP_FN = step
RUN_ONE = partial(run_matrix_sensing_spec, step_fn=STEP_FN)

print(getsource(STEP_FN))

## Plotting API

Stateless plotting functions and the shared color dictionary live in `plotting/`. Each function receives data and returns a Matplotlib figure plus axes; notebook cells below decide when to display each figure.

In [ ]:
from plotting import (
    PLOT_COLORS,
    ordered_algos_in,
    ordered_dims_in,
    plot_algorithm_dimension_grid,
    plot_algorithms_for_dimension,
    plot_all_mean_curves_combined,
    plot_color_key,
    plot_dimensions_for_algorithm,
    plot_metric_bar,
    plot_metric_overview,
    plot_seed_variability_for_dimension,
    summary_table,
)


def show_figure(fig):
    display(fig)
    plt.close(fig)

## Run The Experiment

`run_experiment()` consumes the already materialized `RUN_SPECS`. Each submitted task is `RUN_ONE`, which has `step_fn=STEP_FN` bound in the notebook.

In [ ]:
def build_run_specs():
    return [spec.copy() for spec in RUN_SPECS]


def sort_results(df):
    if df.empty:
        return df
    ordered = df.copy()
    ordered["algo"] = pd.Categorical(ordered["algo"], categories=ALGOS, ordered=True)
    ordered = ordered.sort_values(["algo", "d", "seed"]).reset_index(drop=True)
    ordered["algo"] = ordered["algo"].astype(str)
    return ordered


def run_experiment(run_one=RUN_ONE, num_workers=NUM_WORKERS):
    specs = build_run_specs()
    num_workers = max(1, min(int(num_workers), len(specs)))
    rows = []
    trajectories = {}

    if num_workers == 1:
        for spec in tqdm(specs, desc="E01 runs", unit="run"):
            key, row, traj = run_one(spec)
            rows.append(row)
            trajectories[key] = traj
        return sort_results(pd.DataFrame(rows)), trajectories

    with ProcessPoolExecutor(max_workers=num_workers, mp_context=get_context("spawn")) as executor:
        futures = [executor.submit(run_one, spec) for spec in specs]
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"E01 runs ({num_workers} proc)", unit="run"):
            key, row, traj = future.result()
            rows.append(row)
            trajectories[key] = traj

    return sort_results(pd.DataFrame(rows)), trajectories

## Execute

Run this cell to start the experiment. It leaves results in memory only:

- `df`: one row per run
- `trajectories`: per-step loss and gradient norm

In [ ]:
df, trajectories = run_experiment()

## Result Tables

In [ ]:
summary = summary_table(df)
display(df.sort_values(["algo", "d", "seed"]).reset_index(drop=True))
display(summary)

## Color Key

In [ ]:
fig, axes = plot_color_key(df)
show_figure(fig)

## Metric Overview

In [ ]:
fig, axes = plot_metric_overview(df)
show_figure(fig)

## K Epsilon Bar

In [ ]:
fig, ax = plot_metric_bar(df, "K_epsilon_mean", "Mean K_epsilon by algorithm and dimension", "iterations")
show_figure(fig)

## Wall Clock Bar

In [ ]:
fig, ax = plot_metric_bar(df, "time_s_mean", "Mean wall-clock by algorithm and dimension", "seconds")
show_figure(fig)

## Minimum Loss Bar

In [ ]:
fig, ax = plot_metric_bar(df, "min_loss_mean", "Mean minimum loss by algorithm and dimension", "loss", log_y=True)
show_figure(fig)

## Final Loss Bar

In [ ]:
fig, ax = plot_metric_bar(df, "final_loss_mean", "Mean final loss by algorithm and dimension", "loss", log_y=True)
show_figure(fig)

## Same Dimension Algorithm Comparisons

One figure is produced for each `d`.

In [ ]:
for d in ordered_dims_in(df):
    fig, ax = plot_algorithms_for_dimension(trajectories, d)
    show_figure(fig)

## Same Algorithm Dimension Comparisons

One figure is produced for each algorithm.

In [ ]:
for algo in ordered_algos_in(df):
    fig, ax = plot_dimensions_for_algorithm(trajectories, algo)
    show_figure(fig)

## All Mean Curves

In [ ]:
fig, ax = plot_all_mean_curves_combined(trajectories)
show_figure(fig)

## Algorithm-Dimension Grid

In [ ]:
fig, axes = plot_algorithm_dimension_grid(trajectories)
show_figure(fig)

## Seed Variability By Dimension

One figure is produced for each `d`.

In [ ]:
for d in ordered_dims_in(df):
    fig, ax = plot_seed_variability_for_dimension(trajectories, d)
    show_figure(fig)

## Conclusion

This cell derives a short conclusion from the in-memory result table.

In [ ]:
def conclusion_markdown(df):
    summary = summary_table(df)
    lines = []
    lines.append("### Result Summary")
    lines.append("")
    lines.append(f"- Runs: `{len(df)}`")
    lines.append(f"- Methods: `{', '.join(sorted(df['algo'].unique()))}`")
    lines.append(f"- Iterations per run: `{int(df['iters'].iloc[0])}`")
    lines.append(f"- Total optimizer steps: `{len(df) * int(df['iters'].iloc[0])}`")
    lines.append("")

    for d in sorted(df["d"].unique()):
        sub = summary[summary["d"] == d]
        best_k = sub.loc[sub["K_epsilon_mean"].idxmin()]
        best_time = sub.loc[sub["time_s_mean"].idxmin()]
        best_loss = sub.loc[sub["min_loss_mean"].idxmin()]
        lines.append(
            f"- d={d}: best K_epsilon is `{best_k.algo}` at `{best_k.K_epsilon_mean:.1f}` steps; "
            f"fastest wall-clock is `{best_time.algo}` at `{best_time.time_s_mean:.3f}s`; "
            f"lowest min_loss is `{best_loss.algo}` at `{best_loss.min_loss_mean:.3e}`."
        )

    if {"Muon", "Muon-Exact"}.issubset(set(df["algo"])):
        lines.append("")
        lines.append("Muon vs Muon-Exact checks whether the Newton-Schulz approximation tracks the exact SVD polar direction.")
    if "Shampoo" in set(df["algo"]):
        lines.append("Shampoo is included as a second-order preconditioned baseline; compare both K_epsilon and wall-clock, because its eigendecompositions are not free.")
    return "\n".join(lines)


display(Markdown(conclusion_markdown(df)))